In [10]:
import pandas as pd
print("Environment is ready!")

Environment is ready!


Web Scraping
Use google-play-scraper to collect reviews, ratings, dates, and app names for all three banks.
Target a minimum of 400+ reviews per bank (1,200 total). If the library returns fewer, expand the date range or document the limitation clearly.
Collect the following fields: review text, rating (1–5), review date, bank / app name, source ("Google Play").


# installing Core libraries


In [12]:
# Core libraries
import pandas as pd
import numpy as np
import re
from datetime import datetime

# The star of the show
from google_play_scraper import app, reviews, Sort

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [13]:
# The unique identifier for CBE's app on the Google Play Store
CBE_ID = 'com.combanketh.mobilebanking'

# Step 1: Get app metadata (rating, installs, description...)
app_info = app(
    CBE_ID,
    lang='en',    # Language: English
    country='et'  # Country: Ethiopia
)

print("=" * 50)
print("CBE App Info")
print("=" * 50)
print(f"App Title   : {app_info['title']}")
print(f"Current Score: {app_info['score']}")
print(f"Total Ratings: {app_info['ratings']:,}")
print(f"Total Reviews: {app_info['reviews']:,}")
print(f"Installs     : {app_info['installs']}")

CBE App Info
App Title   : Commercial Bank of Ethiopia
Current Score: 4.284456
Total Ratings: 48,279
Total Reviews: 9,305
Installs     : 5,000,000+


In [14]:
# Step 2: Scrape reviews
print(f"Scraping reviews for CBE...")

result, continuation_token = reviews(
    CBE_ID,
    lang='en',
    country='et',
    sort=Sort.NEWEST,       # Most recent first
    count=450,              # Ask for more than 400 to be safe
    filter_score_with=None  # All star ratings
)

print(f"Collected {len(result)} raw reviews")

Scraping reviews for CBE...
Collected 450 raw reviews


In [15]:
# Step 3: Extract required fields from reviews
cbe_reviews_data = []

for review in result:
    cbe_reviews_data.append({
        'review_text': review.get('reviewText', ''),      # Review content
        'rating': review.get('score', 0),                 # Rating (1-5)
        'review_date': review.get('at', ''),              # Review date
        'bank': 'CBE Bank',                               # Bank/app name
        'app_id': CBE_ID,                                 # App identifier
        'source': 'Google Play'                           # Source
    })

cbe_df = pd.DataFrame(cbe_reviews_data)

print("\n" + "=" * 50)
print("CBE Reviews DataFrame")
print("=" * 50)
print(f"Total reviews collected: {len(cbe_df)}")
print(f"\nColumns: {list(cbe_df.columns)}")
print(f"\nFirst 3 reviews:")
print(cbe_df[['bank', 'rating', 'review_date', 'source']].head(3))
print(f"\nRating distribution:")
print(cbe_df['rating'].value_counts().sort_index())



CBE Reviews DataFrame
Total reviews collected: 450

Columns: ['review_text', 'rating', 'review_date', 'bank', 'app_id', 'source']

First 3 reviews:
       bank  rating         review_date       source
0  CBE Bank       5 2026-05-13 02:19:17  Google Play
1  CBE Bank       2 2026-05-13 01:27:52  Google Play
2  CBE Bank       5 2026-05-13 01:21:48  Google Play

Rating distribution:
rating
1     65
2     10
3     31
4     37
5    307
Name: count, dtype: int64


In [17]:
# Step 4: Prepare final dataset (CBE Bank only)

print(f"\n{'=' * 50}")
print("FINAL DATASET - CBE BANK")
print(f"{'=' * 50}")
print(f"Total reviews: {len(cbe_df)}")
print(f"\nColumns: {list(cbe_df.columns)}")
print(f"\nDataset info:")
print(cbe_df.info())
print(f"\nRating distribution:")
print(cbe_df['rating'].value_counts().sort_index())



FINAL DATASET - CBE BANK
Total reviews: 450

Columns: ['review_text', 'rating', 'review_date', 'bank', 'app_id', 'source']

Dataset info:
<class 'pandas.DataFrame'>
RangeIndex: 450 entries, 0 to 449
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   review_text  450 non-null    str           
 1   rating       450 non-null    int64         
 2   review_date  450 non-null    datetime64[us]
 3   bank         450 non-null    str           
 4   app_id       450 non-null    str           
 5   source       450 non-null    str           
dtypes: datetime64[us](1), int64(1), str(4)
memory usage: 21.2 KB
None

Rating distribution:
rating
1     65
2     10
3     31
4     37
5    307
Name: count, dtype: int64
